In [0]:
# The purpose of this notebook is to export the citibike station status silver table to an Amazon S3 bucket
# that is accessible to the free edition databricks workspaces, so that the computationally expensive task of model
# training can occur in those spaces. It may be that within a few weeks this song and dance is no longer necessary, but as of
# 10/04/2026 the main course databricks workspace is not yet free.
# 
#
# This notebook copies the silver tables to the S3 bucket, for the Free Edition workspace.
#
# The copy of station status in S3 is split into one file per New York day:
#   s3://cuny-sps-citibike-archive/exports/station_status/nyc_date=YYYY-MM-DD/data.parquet
# Each run uploads the complete days not yet in S3. A day already in S3 is never
# rewritten, so Free Edition only needs to download the day folders it doesn't have yet.
#
# The copy of station information is small, so it's replaced in full on every run:
#   s3://cuny-sps-citibike-archive/exports/station_info.parquet
#
# Dev runs write the files to the scratch exports volume and upload nothing.
# Production runs need an AWS key with write access to exports/ only, stored
# as Databricks secrets (scope citibike-export).
from datetime import date, timedelta
from pathlib import Path

# boto3 is an official python package for working with AWS services

import boto3

from pyspark.sql import functions as F

import pipeline_config as cfg

RUN_MODE = cfg.get_run_mode(dbutils)
UPLOAD = RUN_MODE == cfg.PRODUCTION

# Dev reads production but doesn't upload. Production uploads
STATUS_TABLE = f"{cfg.PRODUCTION_SCHEMA}.silver_station_status"
INFO_TABLE = f"{cfg.PRODUCTION_SCHEMA}.silver_station_info"

# The way this works (because you may need to do this yourself depending on how the semester goes)
# is that databricks writes the files to a staging directory. This is technically hosted on S3, but not
# in a bucket that will be accessible to free edition (maybe a cleverer one of you can figure out how to do this
# but I could not). Instead, I have a separate S3 bucket for sharing data to the free edition, and we will use
# boto to write there

# This is where we will stage our upload
STAGING = cfg.volume_path(RUN_MODE, "exports", "staging")

BUCKET = "cuny-sps-citibike-archive"
PREFIX = "exports"
LOCAL_TZ = "America/New_York"
INTERNAL_COLUMNS = ["_source_file", "_rescued_data", "_ingested_at", "_silver_processed_at"]

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Saying what the run is going to do
cfg.describe(RUN_MODE=RUN_MODE, UPLOAD=UPLOAD, STATUS_TABLE=STATUS_TABLE,
             INFO_TABLE=INFO_TABLE, STAGING=STAGING, BUCKET=f"s3://{BUCKET}/{PREFIX}")

# We have stored the credentials for our S3 bucket in databricks. citibike-export is the "scope"

if UPLOAD:
    s3 = boto3.client(
        "s3",
        aws_access_key_id=dbutils.secrets.get("citibike-export", "aws-access-key-id"),
        aws_secret_access_key=dbutils.secrets.get("citibike-export", "aws-secret-access-key"),
        region_name="us-west-2",
    )


# Our folder names have the format nyc_date=YYYY-MM-DD . We will use that to check how many days
# have been uploaded, this makes this task "self logging" (compare with how we had checkpoint directories for our other
# sequential data processing tasks)

def exported_days():
    # In dev we don't upload and also we return the empty "set". So that means that dev will go ahead and 
    # rewrite the entire archive to scratch every time, meaning it will be more expensive and longer to run that
    # production
    if not UPLOAD:
        return set()
    # Start with the empty set
    days = set()

    # "list_objects_v2" is an operation on AWS that lists the
    # files in a given bucket. The paginator is necessary because
    # there is a limit to 1000 files normally, the paginator checks to see
    # if the limit has been hit and then goes back for more. Not relevant yet but
    # one day we may have more than 1000 days of data (future semesters!)
    pages = s3.get_paginator("list_objects_v2").paginate(
        Bucket=BUCKET, Prefix=f"{PREFIX}/station_status/", Delimiter="/")
    
    # Here we extract the dates from the file names. Because of the paginator there are "pages" of results
    # We add each unique day to the set
    for page in pages:
        for folder in page.get("CommonPrefixes", []):
            days.add(folder["Prefix"].rstrip("/").split("nyc_date=")[-1])
    return days


# A day is complete once silver has polls from after its last midnight. This handles
# the challenge of the last upload being a bit fuzzy and unpredictable

local_date = F.to_date(F.from_utc_timestamp("fetched_at", LOCAL_TZ))  # This is a spark formula computes the date the local time date of each poll
polled_days = sorted(row.nyc_date for row in
                     spark.table(STATUS_TABLE).filter(F.col("fetched_at").isNotNull())
                     .select(local_date.alias("nyc_date")).distinct().collect())
complete_days = polled_days[:-1]   # the newest day is treated as unfinished, might not strictly be true but won't cause a problem because
# this is for training, but this could be a problem if the preceding silver had some sort of failure, Might be best to chain this with the
# silver run

days_in_s3 = exported_days()
new_days = [day for day in complete_days if day.isoformat() not in days_in_s3]
print(f"Newest day in silver: {polled_days[-1]} (treated as unfinished); "
      f"{len(days_in_s3)} days already in S3; {len(new_days)} new days to export")



# Check silver data quality before exporting: it must have been updated recently (production only,
# since dev runs can happen mid-week), and the days to export must have unique keys.
silver = spark.table(STATUS_TABLE)
if UPLOAD:
    assert not silver.filter("fetched_at >= current_timestamp() - INTERVAL 6 HOURS").isEmpty(), \
        "No silver polls in the last 6 hours; did the silver job run?"
if new_days:
    assert silver.withColumn("nyc_date", local_date) \
        .filter(F.col("nyc_date").isin([day.isoformat() for day in new_days])) \
        .groupBy("station_id", "fetched_at").count().filter("count > 1").isEmpty(), \
        "Duplicate station/poll keys in the days to export"


# Station status: write one parquet per new day to staging
if new_days:
    status = (
        spark.table(STATUS_TABLE)
        .drop(*INTERNAL_COLUMNS, "snapshot_date")
        .withColumn("nyc_date", F.to_date(F.from_utc_timestamp("fetched_at", LOCAL_TZ)))
        .filter(F.col("nyc_date").isin([day.isoformat() for day in new_days]))
    )
    (status.repartition("nyc_date").write.mode("overwrite") 
           .partitionBy("nyc_date").parquet(f"{STAGING}/station_status"))

# Station information: the full history, one file, replaced each run.
# Written by Spark (not pandas), so timestamps are stored in a form Spark can read back.
# See the station_info structure
nested = ["eightd_station_services", "rental_methods", "rental_uris"]
info = (spark.table(INFO_TABLE).drop(*INTERNAL_COLUMNS)
        .withColumns({column: F.to_json(column) for column in nested}))
info.coalesce(1).write.mode("overwrite").parquet(f"{STAGING}/station_info")


def single_parquet(directory):
    """The one data file Spark wrote in a directory (Spark names it part-....parquet)."""
    files = [p for p in Path(directory).glob("part-*.parquet")]
    if len(files) != 1:
        raise RuntimeError(f"Expected one Parquet file in {directory}, found {len(files)}")
    return files[0]


# Upload to S3. Day files get a fixed name, data.parquet.
uploads = [(single_parquet(f"{STAGING}/station_info"), f"{PREFIX}/station_info.parquet")]
uploads += [(single_parquet(f"{STAGING}/station_status/nyc_date={day.isoformat()}"),
             f"{PREFIX}/station_status/nyc_date={day.isoformat()}/data.parquet")
            for day in new_days]

for local_file, key in uploads:
    if UPLOAD:
        s3.upload_file(str(local_file), BUCKET, key)
        print("uploaded", key)
    else:
        print("dev, not uploaded:", key)

print(f"done: {len(new_days)} new days, station information refreshed")

In [0]:
staged = spark.read.parquet(f"{STAGING}/station_status").groupBy("nyc_date").count()
silver = (spark.table(STATUS_TABLE)
          .select(local_date.alias("nyc_date")).groupBy("nyc_date").count()
          .withColumnRenamed("count", "silver_count"))
comparison = (staged.join(silver, "nyc_date", "full")
              .withColumn("match", F.col("count") == F.col("silver_count"))
              .orderBy("nyc_date"))
display(comparison)

print("Station info rows staged:", spark.read.parquet(f"{STAGING}/station_info").count(),
      "| in silver:", spark.table(INFO_TABLE).count())